In [1]:
pip install ultralytics opencv-python numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.4 MB/s eta 0:00:00


In [2]:
import cv2
from ultralytics import YOLO
import numpy as np
import os

VIDEO_PATH  = "/content/video.mp4"
OUTPUT_PATH = "result.mp4"
ZONE_CAP    = 20         # max people in frame before HIGH crowd level

# ─────────────────────────────────────────────
#  STEP 1 — Load model
# ─────────────────────────────────────────────
print("[1/4] Loading YOLOv8 model...")
model = YOLO("yolov8n.pt")
print("      Model ready.")


print("[2/4] Opening video...")
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(
        f"Cannot open '{VIDEO_PATH}'. "
        "Make sure the file is inside the mall_crowd/ folder."
    )

fps    = cap.get(cv2.CAP_PROP_FPS) or 25
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"      {width}x{height}  {fps:.1f} fps  {total} frames")

os.makedirs("output", exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[1/4] Loading YOLOv8 model...
      Model ready.
[2/4] Opening video...
      640x360  29.9 fps  327 frames


In [3]:
COLORS = {
    "LOW":    (94,  197, 34),    # green   (BGR order — cv2 expects B,G,R)
    "MEDIUM": (60,  146, 251),   # orange  (BGR order)
    "HIGH":   (68,  68,  239),   # red     (BGR order)
}

def get_crowd_level(count, capacity=ZONE_CAP):
    ratio = count / capacity
    if ratio < 0.3:
        return "LOW"
    elif ratio < 0.7:
        return "MEDIUM"
    else:
        return "HIGH"

def apply_protocol(crowd_level):
    table = {
        "LOW":    {"lighting": "40% - economy",   "hvac": "18C - reduced airflow"},
        "MEDIUM": {"lighting": "75% - standard",  "hvac": "21C - normal airflow"},
        "HIGH":   {"lighting": "100% - full",      "hvac": "19C - max airflow + boost"},
    }
    return table[crowd_level]

def _rounded_panel_mask(shape, x, y, w, h, r):
    """Builds a rounded-rectangle mask so the overlay card has soft corners."""
    mask = np.zeros(shape[:2], dtype=np.uint8)
    cv2.rectangle(mask, (x + r, y), (x + w - r, y + h), 255, -1)
    cv2.rectangle(mask, (x, y + r), (x + w, y + h - r), 255, -1)
    for cx, cy in [(x + r, y + r), (x + w - r, y + r),
                   (x + r, y + h - r), (x + w - r, y + h - r)]:
        cv2.circle(mask, (cx, cy), r, 255, -1)
    return mask

def draw_overlay(frame, total_count, crowd_level, cmd):
    """Draws a single, clean status card: count, density level, and protocol."""
    color = COLORS[crowd_level]
    px, py, pw, ph, r = 18, 18, 300, 150, 14

    # semi-transparent dark card with rounded corners
    mask  = _rounded_panel_mask(frame.shape, px, py, pw, ph, r)
    panel = np.full_like(frame, (24, 24, 24))
    frame[:] = np.where(mask[..., None] > 0,
                         cv2.addWeighted(frame, 0.30, panel, 0.70, 0),
                         frame)

    # left accent strip colored by crowd level
    accent = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.rectangle(accent, (px, py + r), (px + 5, py + ph - r), 255, -1)
    frame[accent > 0] = color

    x_label, x_value = px + 24, px + 95

    cv2.putText(frame, "CROWD MONITOR", (x_label, py + 26),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (150, 150, 150), 1, cv2.LINE_AA)

    cv2.putText(frame, f"{total_count}", (x_label, py + 64),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
    (tw, _), _ = cv2.getTextSize(f"{total_count}", cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)
    cv2.putText(frame, "people detected", (x_label + tw + 10, py + 64),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (190, 190, 190), 1, cv2.LINE_AA)

    cv2.circle(frame, (x_label + 5, py + 86), 5, color, -1)
    cv2.putText(frame, f"{crowd_level} DENSITY", (x_label + 18, py + 91),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)

    cv2.line(frame, (x_label, py + 104), (px + pw - 20, py + 104),
             (70, 70, 70), 1, cv2.LINE_AA)

    cv2.putText(frame, "Lighting", (x_label, py + 124),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (150, 150, 150), 1, cv2.LINE_AA)
    cv2.putText(frame, cmd["lighting"], (x_value, py + 124),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220, 220, 220), 1, cv2.LINE_AA)

    cv2.putText(frame, "HVAC", (x_label, py + 144),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (150, 150, 150), 1, cv2.LINE_AA)
    cv2.putText(frame, cmd["hvac"], (x_value, py + 144),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220, 220, 220), 1, cv2.LINE_AA)


In [4]:
print("[3/4] Processing frames...")
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    if frame_idx % 30 == 0:
        pct = frame_idx / total * 100 if total > 0 else 0
        print(f"      Frame {frame_idx}/{total}  ({pct:.0f}%)")

    # Detect persons (class 0)
    results = model(frame, classes=[0], verbose=False)
    boxes   = results[0].boxes

    total_count = len(boxes)
    crowd_level = get_crowd_level(total_count)
    cmd         = apply_protocol(crowd_level)

    # Draw person bounding boxes
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, f"{conf:.0%}",
                    (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX,
                    0.4, (0,255,0), 1, cv2.LINE_AA)

    # Draw the crowd-monitor card (count + density + protocol)
    draw_overlay(frame, total_count, crowd_level, cmd)

    out.write(frame)

cap.release()
out.release()

print(f"[4/4] Done!  Saved → {OUTPUT_PATH}")
print(f"      Processed {frame_idx} frames.")


[3/4] Processing frames...
      Frame 30/327  (9%)
      Frame 60/327  (18%)
      Frame 90/327  (28%)
      Frame 120/327  (37%)
      Frame 150/327  (46%)
      Frame 180/327  (55%)
      Frame 210/327  (64%)
      Frame 240/327  (73%)
      Frame 270/327  (83%)
      Frame 300/327  (92%)
[4/4] Done!  Saved → result.mp4
      Processed 327 frames.
